# 66장 · 역할별 함수 분리와 호출 비용

역할을 함수로 나누면 한 함수의 결과가 다음 역할의 입력이 됩니다. 답변에 필요한 근거가 인계 과정에서 빠지는지와 추가 호출 수를 확인합니다.

과제 제출 질문을 한 에이전트가 처리한 경우와 근거 담당·답변 담당으로 나눈 경우를 비교합니다. 파이썬 함수로 인계한 질문과 문서 ID를 확인하고 두 실행 기록의 호출 수를 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 근거 수집 함수와 답변 함수의 인계값

다음 역할이 문서 ID만 받으면 원문을 다시 찾아야 합니다. 질문, 근거 원문, 출처 ID를 함께 넘겨 답변의 수치나 위치를 확인할 수 있게 합니다. 여기서는 두 역할을 로컬 함수로 실행합니다.


### 셀 01 · 역할 간 근거 데이터 전달

evidence_role의 반환값에서 answer_role이 실제로 읽는 필드를 확인하세요.

answer_role은 text와 id로 문장을 만듭니다. question은 전달받아도 문장을 작성할 때 사용하지 않습니다.


In [1]:
def evidence_role(question):
    return {"question": question, "id": "course-guide-v1:p2",
            "text": "과제는 학습 관리 페이지의 과제함에 제출합니다."}

def answer_role(handoff):
    return f"{handoff['text']} [{handoff['id']}]"

handoff = evidence_role("과제는 어디에 내나요?")
print(answer_role(handoff))

과제는 학습 관리 페이지의 과제함에 제출합니다. [course-guide-v1:p2]


답변 끝의 p2는 근거 담당이 전달한 ID입니다. 문장을 만드는 쪽에서 새로 붙인 문서 번호가 아닙니다.

두 역할 사이에서 최종 문장만 전달하면 두 번째 역할은 p2의 어느 구절을 읽었는지 확인하기 어렵습니다. 원래 질문과 근거 ID를 함께 넘겨야 인계 전후의 요청이 같은지, 답변이 같은 근거를 사용했는지 대조할 수 있습니다.


#### 두 함수가 같은 근거를 넘기는 위치

```mermaid
flowchart TD
ninput["질문"]
  nlookup["evidence_role"]
  nhandoff["인계 객체"]
  nwrite["answer_role"]
  ninput --> nlookup
  nlookup --> nhandoff
  nhandoff --> nwrite
```

이 셀은 모델을 호출하지 않습니다. 근거 함수가 만든 딕셔너리에서 답변 함수가 어떤 값을 이어받는지 볼 수 있습니다.

### 두 역할 사이의 인계값

<p>근거 담당이 넘긴 question·id·text 중 답변 함수가 쓰는 값은 text와 id입니다. 새 질문의 근거가 맞는지도 확인해 보세요.</p>


### 과제 02 · 인계 시 질문 보존 검사

검색 담당이 받은 원 질문과 인계된 질문이 같은지 확인하세요.


In [2]:
question = '과제는 어디에 내나요?'
# handoff의 질문 필드를 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>question = &#x27;과제는 어디에 내나요?&#x27;
print(question == handoff[&#x27;question&#x27;])</code></pre>
<p>실행 결과</p><pre><code>True</code></pre>
<p>True입니다. 문서가 같더라도 질문이 바뀌면 두 방식의 결과를 같은 과제로 비교하기 어렵습니다.</p>
</details>


#### 역할을 나눈 뒤 무엇이 달라졌을까요?

<mark>근거 수집 결과에는 다음 함수가 쓸 본문과 출처가 함께 있어야 합니다.</mark> 두 경로의 마지막 문장이 같아도 역할 사이에 넘긴 값과 호출 수는 다를 수 있습니다. 같은 질문으로 비교하고 실제로 추가된 일을 기록하세요.

선수 개념 · 63장 RAG 서비스 함수와 응답 계약

### 질문이 다음 역할까지 같은지 확인하기

<p>인계받은 근거가 다른 질문의 것이라면 형식이 맞아도 답변을 만들 수 없습니다. answer_role을 부르기 전에 handoff["question"]과 원래 질문을 대조합니다.</p>


<h3>역할 사이에 전달되는 딕셔너리</h3>

<p>evidence_role은 질문·문서 ID·원문을 한 객체에 넣고 answer_role은 그중 text와 id로 답을 만듭니다. 두 함수는 모델을 호출하지 않습니다. 뒤의 model_calls 2회와 4회는 셀에 따로 제공한 두 경로의 관찰 기록입니다.</p>


## 단일 경로와 역할 분리 경로 비교

셀 03의 딕셔너리 두 개에는 같은 질문 Q01의 대본 실행 기록이 들어 있습니다. 앞의 파이썬 함수 두 개가 이 횟수를 발생시킨 것은 아닙니다.


### 셀 03 · 역할별 호출 횟수 비교

single과 two_roles의 기록에서 모델 호출 횟수가 얼마나 달라졌는지 계산하세요.


In [3]:
single = {"model_calls": 2, "mandatory_retrieval_calls": 1}
two_roles = {"model_calls": 4, "mandatory_retrieval_calls": 1}
print(two_roles["model_calls"] - single["model_calls"])

2


출력 2는 제공 기록의 4회에서 2회를 뺀 값입니다. 필수 검색은 양쪽 모두 1회입니다. 호출당 토큰이나 단가는 기록에 없어 추가 비용은 계산할 수 없습니다.


### 같은 단위끼리 호출 수 비교하기

<p>두 관찰값 모두 동일 질문에서 필수 검색을 한 번 수행했습니다. model_calls의 차이와 mandatory_retrieval_calls의 차이를 따로 계산하면 무엇이 늘었는지 알 수 있습니다.</p>


### 역할이 늘 때 들인 비용

<p>기록에 적힌 모델 호출은 단일 경로에서 2회, 역할 분리 경로에서 4회입니다. 이 셀은 기록의 차이 2를 계산합니다. 두 모델을 새로 호출해 재측정하지는 않습니다.</p>


## 복습과 실습 점검

### 과제 04 · 인계한 근거가 새 질문에도 맞는지 확인하기

같은 함수에 주차 요금 질문을 넘기고 받은 근거 문장을 확인하세요. 두 번째 줄에서 주차 요금을 확인할 수 있는지 살펴보세요.


In [ ]:
# evidence_role에 "주차비는 얼마인가요?"를 전달해 원문을 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>parking = evidence_role(&quot;주차비는 얼마인가요?&quot;)
print(parking[&quot;question&quot;])
print(parking[&quot;text&quot;])</code></pre>
<p>실행 결과</p><pre><code>주차비는 얼마인가요?
과제는 학습 관리 페이지의 과제함에 제출합니다.</code></pre>
<p>넘겨받은 질문은 정확하지만 근거는 과제 제출 안내입니다. evidence_role이 어떤 질문에도 같은 문서를 반환하기 때문입니다. 이 경우에는 근거를 새로 찾거나 빈 근거로 종료해야 하며, 같은 자료를 다른 역할에 한 번 더 넘겨도 주차비를 알 수 없습니다.</p>
</details>


## 두 역할 사이에서 근거가 잘못 넘어온 지점 찾기

앞의 evidence_role은 모든 질문에 과제 제출 문서를 돌려줬습니다. 답변 역할을 늘려도 이 문제가 남아 있으면 주차 질문에 과제 제출 안내가 나옵니다. 역할 간 전달 형식을 먼저 고치고 오류가 사라지는지 다시 실행합니다.

### 셀 05 · 정상 근거와 보류 상태를 받는 인계 함수 만들기

이 자료에는 과제 제출 방법만 있습니다. 작은 재현 예에서는 질문에 과제와 제출이 함께 있을 때만 근거를 사용합니다. 새로운 주제나 표현을 다룰 때는 이 단어 규칙의 한계도 점검해야 합니다.

In [4]:
def lookup_handoff(question):
    if "과제" not in question or "제출" not in question:
        return {"question": question, "status": "no_evidence", "evidence": None}
    matched = evidence_role(question)
    return {"question": question, "status": "matched", "evidence": matched}

miss = lookup_handoff("주차비는 얼마인가요?")
print(miss)

{'question': '주차비는 얼마인가요?', 'status': 'no_evidence', 'evidence': None}


질문은 보존됐고 상태는 no_evidence, evidence는 None입니다. 여기까지는 조회 역할의 책임입니다. 답변 역할에 다시 자료를 찾게 하기보다, 넘겨받은 상태로 답할 수 있는지 판단하게 합니다.

### 조회 에이전트의 도구 결과가 작성 에이전트에 도착하는 경로

```mermaid
flowchart TD
    research["조회 에이전트의 모델"] --> choice{"새 도구 요청이 있는가?"}
    choice -->|있음| lookup["assignment_source 실행"]
    lookup --> result["ToolMessage를 조회 기록에 추가"]
    result --> research
    choice -->|없음| done["조회 에이전트 종료"]
    done --> handoff["첫 ToolMessage의 content 읽기"]
    handoff --> writer["작성 에이전트의 모델"]
    writer --> finish["도구 호출 없이 답변 반환"]
```

조회 모델이 처음에는 source-1을 요청하고, 도구 결과를 받은 뒤 호출 없는 메시지를 보내면 조회 그래프가 끝납니다. 바깥 코드가 조회 기록에서 첫 ToolMessage의 content를 꺼내 작성 에이전트의 사용자 메시지로 건넵니다. 작성 에이전트에는 도구가 없습니다. 이 예제의 next는 조회 도구가 한 번 실행됐다는 조건에 기대고 있어, 호출을 생략하는 대본에서는 빈 결과를 따로 처리해야 합니다.

### 과제 06 · 답변 역할을 연결해 같은 실패 사례 재실행하기

정상 근거가 있을 때는 기존 answer_role에 evidence를 넘기고, 근거가 없으면 안내를 보류하는 render_handoff를 작성하세요. RunnableLambda 두 개를 연결해 주차 질문과 과제 제출 질문을 각각 보내세요.

In [5]:
from langchain_core.runnables import RunnableLambda
# def render_handoff(packet): ...
# lookup_handoff와 새 함수를 순서대로 연결하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>from langchain_core.runnables import RunnableLambda
def render_handoff(packet):
    if packet[&quot;status&quot;] == &quot;no_evidence&quot;:
        return &quot;관련 안내를 찾지 못했습니다.&quot;
    return answer_role(packet[&quot;evidence&quot;])

service = RunnableLambda(lookup_handoff) | RunnableLambda(render_handoff)
print(service.invoke(&quot;주차비는 얼마인가요?&quot;))
print(service.invoke(&quot;과제를 어디에 제출하나요?&quot;))</code></pre>
<p>실행 결과</p><pre><code>관련 안내를 찾지 못했습니다.
과제는 학습 관리 페이지의 과제함에 제출합니다. [course-guide-v1:p2]</code></pre>
<p>첫 질문은 보류되고 두 번째에는 과제함 안내와 출처 ID가 나옵니다. 두 역할을 각각 호출해 보면 어떤 단계에서 no_evidence가 생겼는지 찾을 수 있습니다. 이 체인은 Python 함수를 연결했으므로 역할의 경계를 확인하는 데 모델 호출 비용이 들지 않습니다.</p>
</details>

### 셀 07 · 한 에이전트의 검색 요청과 근거 확인하기

이번에는 실제 create_agent 그래프를 실행합니다. 검색 요청을 만드는 모델의 응답 순서만 고정하고, 도구는 방금 작성한 조회 함수를 호출합니다. 최종 문장을 읽기 전에 근거가 든 도구 메시지를 따로 꺼내세요.

In [6]:
from langchain.agents import create_agent
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langsmith import tracing_context

@tool
def assignment_source(question: str) -> str:
    """수업 자료에서 과제 제출 안내와 문서 ID를 찾습니다."""
    packet = lookup_handoff(question)
    if packet["evidence"] is None:
        return "자료 없음"
    return answer_role(packet["evidence"])

class ResearchModel(FakeMessagesListChatModel):
    def bind_tools(self, tools, **kwargs):
        return self

query = "과제를 어디에 제출하나요?"
request_source = AIMessage(content="", tool_calls=[{
    "name": "assignment_source", "args": {"question": query}, "id": "source-1",
}])
research_agent = create_agent(ResearchModel(responses=[
    request_source, AIMessage(content="문서에서 제출 방법을 확인했습니다."),
]), tools=[assignment_source])
with tracing_context(enabled=False):
    research_log = research_agent.invoke({"messages": [("user", query)]})["messages"]
evidence_line = next(m.content for m in research_log if m.type == "tool")
print([m.type for m in research_log], evidence_line)

['human', 'ai', 'tool', 'ai'] 과제는 학습 관리 페이지의 과제함에 제출합니다. [course-guide-v1:p2]


한 그래프에서 human → ai → tool → ai가 기록됩니다. 첫 ai는 조회 요청이고 마지막 ai는 조회를 마친 대본의 문장입니다. 다음 역할이 문서 ID까지 확인하려면 마지막 인사말 대신 실제 도구 결과인 evidence_line을 넘겨야 합니다.

### 셀 08 · 두 번째 에이전트가 받아 본 문자열 검산하기

작성 역할은 전달받은 문장을 그대로 반환하는 로컬 테스트 모델에 연결합니다. 두 그래프를 거친 기록에서 실제로 붙은 ai 메시지 수를 세세요. 문장을 다듬는 성능은 이 테스트에서 평가하지 않습니다.

In [7]:
from langchain_core.language_models.chat_models import SimpleChatModel

class ReadBack(SimpleChatModel):
    @property
    def _llm_type(self):
        return "handoff-readback"
    def _call(self, messages, **kwargs):
        return messages[-1].content

writer_agent = create_agent(ReadBack(), tools=[])
with tracing_context(enabled=False):
    writer_log = writer_agent.invoke({"messages": [("user", evidence_line)]})["messages"]
print(writer_log[-1].content)
print(sum(m.type == "ai" for m in research_log),
      sum(m.type == "ai" for m in research_log + writer_log))

과제는 학습 관리 페이지의 과제함에 제출합니다. [course-guide-v1:p2]
2 3


출처 ID가 포함된 문장이 두 번째 역할까지 전달됩니다. 조회 에이전트만 쓴 기록에서는 ai 메시지가 두 개이고 작성 역할까지 더하면 세 개입니다. 이 사례에서 한 번 더 실행할 이유가 있으려면, 실제 서비스의 작성 역할이 근거 검토나 표현 수정 등 어떤 일을 맡을지 먼저 정해야 합니다.

<mark>역할을 둘로 나눌 때는 근거를 못 찾은 경우도 인계 형식에 넣으세요.</mark> 문서 ID만 넘기도록 설계하면 검색이 실패했을 때 답변 역할이 상태를 구별하기 어렵습니다. status가 있으면 응답을 만들기 전에 보류할 수 있습니다.

다음 장에서는 같은 실패 사례를 수정 전후에 반복해 답과 근거를 함께 평가합니다.